> ###Remove all where brnach_id not null
###Remove all the leading and trailing spaces in Brnach Country and covert it into UPPER CASE

In [0]:
df = spark.sql("""
    SELECT * FROM (
        SELECT b.branch_id,b.branch_city,upper(trim(b.branch_country)) as branch_country,
               ROW_NUMBER() OVER (PARTITION BY branch_id ORDER BY branch_id) as rn
        FROM bronzelayer.branch b
        WHERE branch_id IS NOT NULL AND maerge_flag = False
    ) WHERE rn = 1
""")
display(df)


In [0]:
df.createOrReplaceTempView("clean_branch")
spark.sql( "MERGE INTO silverlayer.branch AS T USING clean_branch AS S ON t.branch_id = s.branch_id when MATCHED THEN UPDATE SET t.branch_country = s.branch_country , t.branch_city = s.branch_city, t.merged_timestamp = current_timestamp()  when NOT MATCHED THEN insert (branch_id, branch_country,branch_city,merged_timestamp ) values (s.branch_id, s.branch_country , s.branch_city,current_timestamp())")


In [0]:
%sql
select * from silverlayer.branch

In [0]:
spark.sql("update bronzelayer.agent set merge_flag = True WHERE merge_flag = false")